Github repo: https://github.com/<redacted>/assignment-4-equation-of-a-slime-<redacted>

Commit ID: 77c1fb9d7a80f11874c998a9b309326f6ffd0ad4

In [1]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [2]:
# Using pandas load the dataset (load remotely, not locally)
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)

df = pd.read_csv("science_data_large.csv")
display(df.head(15))
df.info()

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


## Part 2. Splitting the dataset

In [3]:
# Take the pandas dataset and split it into our features (X) and label (y)

x = df[['Temperature °C', 'Mols KCL']]
y = df['Size nm^3']

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.1)

## Part 3. Perform a Linear Regression

In [9]:
# Use sklearn to train a model on the training set
model = LinearRegression().fit(x_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
sample = pd.DataFrame({'Temperature °C': [469], 'Mols KCL': [647]})
size_prediction = model.predict(sample)
print(f"Sample prediction: {size_prediction}")

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
y_prediction = model.predict(x_test)
r_2_score = r2_score(y_test, y_prediction)

print(f"R-squared score of the model: {r_2_score}")

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = pd.DataFrame({
    'Feature': x.columns,
    'Coefficient': model.coef_
})
intercept = model.intercept_
temperature_coefficient = coefficients[coefficients['Feature'] == 'Temperature °C']['Coefficient'].values
mols_coefficient = coefficients[coefficients['Feature'] == 'Mols KCL']['Coefficient'].values

print(temperature_coefficient)
print(mols_coefficient)
print(intercept)

Sample prediction: [659769.59415734]
R-squared score of the model: 0.8539621162180726
[887.36787523]
[1034.50715061]
-425732.0657745847


The score represents the how well the model explains the variance of the size of the slime. A score of 1 means that all variance can be explained, while 0 means it can't be. Our score of .87 means that 87% of the variance can be explained by temperature and KCL concentration,while 13% were due to factors outside the features provided


Equation: $h(x) = 887.37\times x_0 + 1034.5\times x_1 - 425730$

Sample equation: $E = mc^2$

## Part 4. Use Cross Validation

In [5]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
cvs = cross_val_score(LinearRegression(), x, y, cv=10)
print(cvs)
# Report on their finding and their significance

[0.81123596 0.86440978 0.87808742 0.86561069 0.87495621 0.84484397
 0.87941022 0.86349411 0.78353682 0.88686516]


After repeating the experiment 10 times, it looks like the linear regression model fluctuates between 78% to 88%. This shows that the model is pretty consistent and isn't overfitted on the training data. The percentages have the same meaning as the percentages above (variance).

## Part 5. Using Polynomial Regression

In [10]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(2)

poly_x_train = poly.fit_transform(x_train)
poly_x_test = poly.transform(x_test)

poly_model = LinearRegression().fit(poly_x_train, y_train)

poly_y_prediction = poly_model.predict(poly_x_test)
poly_r_2_score = r2_score(y_test, poly_y_prediction)
print(poly_r_2_score)

coefficients_poly = pd.DataFrame({
    'Feature': poly.get_feature_names_out(x.columns),
    'Coefficient': poly_model.coef_
})
intercept = poly_model.intercept_
poly_temperature_coefficient = coefficients_poly.values

print(poly_temperature_coefficient)

poly_intercept = poly_model.intercept_
print(poly_intercept)
# Report on the metrics and output the resultant equation as you did in Part 3.

1.0
[['1' 0.0]
 ['Temperature °C' 11.999999995232228]
 ['Mols KCL' -1.5007503976007708e-07]
 ['Temperature °C^2' -8.967649275640209e-12]
 ['Temperature °C Mols KCL' 2.0000000000430256]
 ['Mols KCL^2' 0.0285714287285616]]
2.0429608412086964e-05


The augmented data seems to have given a score of 1, meaning that all of the variance is explained by the temperature and the concentration of KCL. This means the regression is perfect, which is very surprising! There are more coefficients now, so we have to update the equation. The concentration of KCL is very small, as is the intercept, so we can treat them as 0.

Equation: $h(x) = 12\times x_0 +2\times x_0x_1 + 0.02857 \times x_1^2$